# Lab 5: Convolutional networks

**DSAN-6600 Deep Learning · Fall 2026**

**Due:** Wednesday Oct 7, 11:59 PM ET · **Target time:** about 60 minutes

---

## What this lab is for

Thursday's lecture claimed that a convolutional network beats a fully connected
one on images at the same parameter count. You are going to check that, on the
same Fashion-MNIST data you used in Lab 4, against the same model you built in
Lab 4.

Four parts:

1. **Shape arithmetic**, on paper, before you write any model code. If you
   cannot predict the shape coming out of a `Conv2d`, you will spend your
   career reading stack traces about it.
2. **Build the network**, out of the conv, norm, activation, pool block, with a
   head that does not flatten.
3. **Run the comparison.** Same data, same optimizer, same schedule, same
   number of epochs, parameter counts within 1%. The only difference is the
   architecture.
4. **Augmentation**, which will not do what you expect, and the reason it does
   not is the most useful thing in this lab.

You already wrote the training loop in Lab 4, so this lab gives it to you.
Nothing here asks you to derive anything.

## How this lab is graded

- **The notebook is graded for completion, not for correctness.** Submit it,
  with all cells run and outputs visible, and you get the points.
- **You may work together.** Everyone runs the same seed, so your numbers
  should land close to your neighbor's. They will not match to the last decimal
  if one of you is on a GPU and the other is on a CPU, which is normal.
- **The understanding is assessed on Quiz 6**, the Thursday after the due date.
- **Solutions are posted after the deadline.**

## Using AI on this lab

Allowed and expected, with disclosure; there is a cell for that at the end. An
agent will write this network in seconds. It will also tell you, confidently,
that data augmentation improves accuracy. Part 4 measures that claim on this
dataset. Measure before you believe.

## Step 0: Setup

**The download.** The first cell fetches Fashion-MNIST into a `data/` folder
next to this notebook, about 82 MB. If you still have Lab 4's copy you can
point `DATA` at it instead.

**How long it runs.** Five training runs, about three minutes of compute on a
desktop CPU and under ten on Colab's free tier. No GPU needed; if you have one
the code uses it.

In [ ]:
import time

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets
from torchvision.transforms import v2

SEED = 6600
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA = "data"

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

print(f"torch {torch.__version__} on {DEVICE}")

---

# Part 1 · Shapes, before any model code

A `Conv2d` does not change the number of dimensions, so a shape bug here does
not raise. It gives you a model that trains to something mediocre and a tensor
you cannot explain. The arithmetic is three lines, so learn it now.

## 1a · Output size

For one spatial dimension, with input size $W$, kernel $k$, padding $p$ and
stride $s$:

$$W_{\text{out}} = \left\lfloor \frac{W + 2p - k}{s} \right\rfloor + 1$$

Fill in the three predictions below **before** you run the checking cell. Each
is one line of arithmetic.

In [ ]:
# Input is a Fashion-MNIST batch: 64 images, 1 channel, 28 x 28.
#
# TODO: predict the output shape of each layer, as a tuple (N, C, H, W).
#   Use the formula above for H and W. C is whatever the layer was given as
#   out_channels. N never changes.

# nn.Conv2d(1, 32, kernel_size=3, padding=1), stride 1
pred_a = None

# nn.Conv2d(1, 32, kernel_size=3, padding=0), stride 1
pred_b = None

# nn.Conv2d(1, 32, kernel_size=5, padding=2, stride=2)
pred_c = None

print(pred_a, pred_b, pred_c)

Now check yourself. This is the habit worth keeping: predict, then run.

In [ ]:
x = torch.zeros(64, 1, 28, 28)
layers = {
    "pred_a": nn.Conv2d(1, 32, 3, padding=1),
    "pred_b": nn.Conv2d(1, 32, 3, padding=0),
    "pred_c": nn.Conv2d(1, 32, 5, padding=2, stride=2),
}
guesses = {"pred_a": pred_a, "pred_b": pred_b, "pred_c": pred_c}

for name, layer in layers.items():
    actual = tuple(layer(x).shape)
    mark = "ok  " if guesses[name] == actual else "WRONG"
    print(f"  {mark} {name}: you said {guesses[name]}, it is {actual}")

## 1b · Where the parameters are

A convolutional layer's weights do not depend on the image size at all. That is
the whole point of weight sharing, and it is why the parameter count below
surprises people who are used to `nn.Linear`.

For `Conv2d(C_in, C_out, k)` with a bias, the count is

$$C_{\text{out}} \times (C_{\text{in}} \times k \times k + 1)$$

In [ ]:
# TODO: fill in conv_params so it returns the number of learnable parameters
#   in a Conv2d(c_in, c_out, k) with bias=True. One line, using the formula
#   above.

def conv_params(c_in, c_out, k):
    raise NotImplementedError("conv_params")


for c_in, c_out, k in [(1, 32, 3), (32, 64, 3), (64, 128, 3), (128, 96, 3)]:
    got = conv_params(c_in, c_out, k)
    real = sum(p.numel() for p in nn.Conv2d(c_in, c_out, k).parameters())
    print(f"  Conv2d({c_in:3d}, {c_out:3d}, {k})  you say {got:>7,}  "
          f"actually {real:>7,}  {'ok' if got == real else 'WRONG'}")

Compare the first of those to a fully connected layer that reads the same
28 by 28 image and produces 32 numbers: $785 \times 32 = 25{,}120$ weights.
The convolution uses **320**, and it applies them at all 784 positions instead
of once.

## 1c · Receptive field

A 3 by 3 convolution looks at 3 pixels. Stack a second one and each output
looks at 5. Pool in between and the window grows faster, because every later
pixel covers two earlier ones.

Going forward through the network, with $r$ the receptive field so far and $j$
the jump (how many input pixels one step in this feature map covers):

- a `Conv2d` with kernel $k$: $r \leftarrow r + (k - 1) \times j$
- a `MaxPool2d(2)`: $r \leftarrow r + j$, then $j \leftarrow 2j$

In [ ]:
# The first half of the network you are about to build:
#   Conv2d(1, 32, 3)  ->  MaxPool2d(2)  ->  Conv2d(32, 64, 3)  ->  MaxPool2d(2)
#
# TODO: track r and j through those four layers by hand and fill in the final
#   value of r. Start at r = 1, j = 1. Use the two update rules above.
#   Write the four intermediate values in the list so you can see your work.

r_steps = []   # r after each of the four layers
rf_final = None

print(f"receptive field after each layer: {r_steps}")
print(f"final receptive field: {rf_final} pixels of the 28 x 28 input")

Ten pixels of twenty-eight, a third of the image, after four layers. A sleeve
fits in that window; a whole coat does not. The two later convolutions, which
run at a jump of 4, each add 8 more and take it to 26. Only the global average
pool at the end sees everything at once.

---

# Part 2 · Build the network

## 2a · One block

Essentially every convolutional network is built from the same small block,
repeated:

```
Conv2d  ->  BatchNorm2d  ->  ReLU  ->  (MaxPool2d)
```

The order matters and is not arbitrary. The convolution is linear, so
normalizing after it is normalizing the thing that actually drifts during
training. The activation comes after the normalization, because normalizing a
ReLU output would just undo the zeroing. Pooling comes last, and only in some
blocks.

`BatchNorm2d` takes the **channel count**, not the spatial size, because it
keeps one mean and one variance per channel.

In [ ]:
def conv_block(c_in, c_out, pool):
    """Conv -> BatchNorm -> ReLU, and a 2x2 max pool if pool is True.

    Returns a list of layers, so that nn.Sequential(*block(...), *block(...))
    builds a flat network.
    """
    # TODO: build and return the list of layers.
    #   Conv2d: 3x3 kernel, padding=1 so the spatial size is unchanged.
    #   BatchNorm2d: takes c_out.
    #   ReLU, then MaxPool2d(2) only if pool is True.
    raise NotImplementedError("conv_block")


layers = conv_block(1, 32, pool=True)
print([type(l).__name__ for l in layers])
print(f"no-pool block: {[type(l).__name__ for l in conv_block(32, 64, pool=False)]}")

## 2b · The whole network

Four blocks, then a head. The channel counts go up as the spatial size comes
down, which is the standard shape of a convolutional network: you trade
resolution for the number of things you can describe at each position.

| Block | Channels | Pool | Spatial size after |
|---|---|---|---|
| 1 | 1 to 32 | yes | 14 by 14 |
| 2 | 32 to 64 | yes | 7 by 7 |
| 3 | 64 to 128 | no | 7 by 7 |
| 4 | 128 to 96 | no | 7 by 7 |

Then the head, which is the part worth paying attention to:

```
AdaptiveAvgPool2d(1)  ->  Flatten  ->  Linear(96, 10)
```

`AdaptiveAvgPool2d(1)` averages each channel down to a single number, turning
`(N, 96, 7, 7)` into `(N, 96, 1, 1)`. The classifier then reads 96 numbers,
one per channel, each one answering "how much of this feature was anywhere in
the image".

In [ ]:
def build_cnn():
    torch.manual_seed(SEED)
    # TODO: return the nn.Sequential described in the table above.
    #   Four conv_block calls, the first two with pool=True.
    #   Then nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(96, 10).
    #   Unpack each block with * so the result is one flat Sequential.
    raise NotImplementedError("build_cnn")


cnn = build_cnn()
n_cnn = sum(p.numel() for p in cnn.parameters())
print(cnn)
print(f"\nparameters: {n_cnn:,}")
print(f"output shape on a batch of 64: {tuple(cnn(torch.zeros(64,1,28,28)).shape)}")

Watch the shape come down through the network. Nothing here is a mystery once
you can do Part 1a.

In [ ]:
h = torch.zeros(2, 1, 28, 28)
print(f"  {'input':24s} {tuple(h.shape)}")
for layer in cnn:
    h = layer(h)
    print(f"  {type(layer).__name__:24s} {tuple(h.shape)}")

## 2c · Why the head does not flatten

The alternative head is the one you would write first: flatten the
`(N, 96, 7, 7)` feature map and put a `Linear` on it. It works. It is also
where almost all of your parameters would go.

In [ ]:
# TODO: how many parameters would Linear(96*7*7, 10) have, with bias?
#   A Linear(in, out) has out * (in + 1) parameters.

flatten_head = None

gap_head = sum(p.numel() for p in nn.Linear(96, 10).parameters())
print(f"  flatten then Linear : {flatten_head:>7,} parameters")
print(f"  AdaptiveAvgPool2d   : {gap_head:>7,} parameters")
print(f"  ratio               : {flatten_head / gap_head:>7.1f}x")

Forty-eight times the parameters for the same ten outputs, and the flatten head
on its own would be about a fifth of the entire network.

It also locks the model to one input size. Change the image from 28 by 28 and
`96*7*7` is the wrong number, so the layer has to be rebuilt. Average pooling
takes any input size large enough to survive the pooling.

This is why every architecture after VGG uses global average pooling.

---

# Part 3 · The comparison

## 3a · The data, shaped for a convolution

Lab 4's `Dataset` flattened each image to `(784,)` because `nn.Linear` wanted a
vector. A convolution wants the opposite: the spatial layout is the whole point,
so the image must stay `(C, H, W)`.

One other change from Lab 4. This `Dataset` takes an optional `transform` and
applies it in `__getitem__`, which is where augmentation belongs. Part 4 uses
it; for now it is `None`.

In [ ]:
class FashionImages(Dataset):
    """uint8 (28, 28) in -> float32 (1, 28, 28) in [0, 1] out."""

    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        # TODO: return (x, y) for example i.
        #   1. take self.images[i], which is uint8 of shape (28, 28)
        #   2. add the channel axis so it is (1, 28, 28). .unsqueeze(0) does it
        #   3. if self.transform is not None, apply it. Transforms want uint8
        #      CHW, which is what you now have, so pass it straight through
        #   4. cast to float and scale to [0, 1], then return with the label
        #      unchanged. Do NOT one-hot encode the label
        raise NotImplementedError("__getitem__")

In [ ]:
raw = datasets.FashionMNIST(DATA, train=True, download=True)
CLASSES = raw.classes

# Same split as lecture: one shuffle with the shared seed, 12,000 for training
# and 2,000 held back. Fashion-MNIST ships a test set too, which this lab does
# not touch at all.
g = torch.Generator().manual_seed(SEED)
order = torch.randperm(len(raw), generator=g)[:14_000]
x_all, y_all = raw.data[order], raw.targets[order]
x_train, y_train = x_all[:12_000], y_all[:12_000]
x_val, y_val = x_all[12_000:], y_all[12_000:]

ds = FashionImages(x_train, y_train)
xb, yb = ds[0]
print(f"train {tuple(x_train.shape)}   val {tuple(x_val.shape)}")
print(f"one example: {tuple(xb.shape)} {xb.dtype}, "
      f"values {xb.min():.2f} to {xb.max():.2f}, label {yb} ({CLASSES[yb]})")

## 3b · Lab 4's model, and the training loop

This is the model from Lab 4, unchanged: 203,530 parameters. Note the
`nn.Flatten()` on the front, which is now doing real work, because the data
arrives as `(N, 1, 28, 28)` instead of `(N, 784)`.

In [ ]:
def build_mlp():
    torch.manual_seed(SEED)
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(784, 256), nn.ReLU(),
        nn.Linear(256, 10),
    )


n_mlp = sum(p.numel() for p in build_mlp().parameters())
print(f"  MLP : {n_mlp:,} parameters")
print(f"  CNN : {n_cnn:,} parameters")
print(f"  the CNN has {100 * (n_cnn / n_mlp - 1):+.1f}% more")

The training loop is Lab 4's, so it is given to you. Two things in it are worth
a second look, and both are from Thursday.

`model.train()` and `model.eval()` do not switch gradients on and off. What
they switch is the behavior of `BatchNorm2d` and `Dropout`. In `train()` mode
BatchNorm normalizes using the current batch's statistics; in `eval()` mode it
uses a running average collected during training. Part 3d measures the
difference, so `evaluate` below can be asked for either.

In [ ]:
def evaluate(model, loader, train_mode=False):
    """Validation accuracy and loss. train_mode=True leaves BatchNorm using
    the batch's own statistics instead of the running estimates."""
    model.train() if train_mode else model.eval()
    lossf = nn.CrossEntropyLoss(reduction="sum")
    total_loss = correct = n = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = model(xb)
            total_loss += lossf(logits, yb).item()
            correct += (logits.argmax(1) == yb).sum().item()
            n += len(yb)
    return 100 * correct / n, total_loss / n


def train(model, train_ds, val_ds, epochs=18, batch_size=128, track_modes=False):
    """Lab 4's loop: AdamW, cosine schedule, cross entropy on logits."""
    g = torch.Generator().manual_seed(SEED)
    tl = DataLoader(train_ds, batch_size=batch_size, shuffle=True, generator=g)
    vl = DataLoader(val_ds, batch_size=512)

    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    lossf = nn.CrossEntropyLoss()

    hist = {"acc": [], "loss": [], "acc_train_mode": []}
    t0 = time.time()
    for ep in range(epochs):
        model.train()
        for xb, yb in tl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = lossf(model(xb), yb)
            loss.backward()
            opt.step()
        sched.step()

        acc, vloss = evaluate(model, vl)
        hist["acc"].append(acc)
        hist["loss"].append(vloss)
        if track_modes:
            hist["acc_train_mode"].append(evaluate(model, vl, train_mode=True)[0])
    hist["seconds"] = time.time() - t0
    return model, hist

## 3c · Run both

Same data, same optimizer, same learning rate, same cosine schedule, same 18
epochs, parameter counts within 1%. The architecture is the only thing that
differs.

This is the slowest cell in the lab, about a minute and a half on a desktop CPU.

In [ ]:
train_ds = FashionImages(x_train, y_train)
val_ds = FashionImages(x_val, y_val)

mlp, h_mlp = train(build_mlp(), train_ds, val_ds)
print(f"  MLP  final {h_mlp['acc'][-1]:.2f}%   ({h_mlp['seconds']:.0f}s)")

cnn, h_cnn = train(build_cnn(), train_ds, val_ds, track_modes=True)
print(f"  CNN  final {h_cnn['acc'][-1]:.2f}%   ({h_cnn['seconds']:.0f}s)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ep = range(1, len(h_mlp["acc"]) + 1)
for h, lab, col in [(h_mlp, f"MLP ({n_mlp:,} params)", "#7f8c8d"),
                    (h_cnn, f"CNN ({n_cnn:,} params)", "#1c7a3e")]:
    axes[0].plot(ep, h["loss"], "-o", ms=4, color=col, label=lab)
    axes[1].plot(ep, h["acc"], "-o", ms=4, color=col, label=lab)
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("validation loss")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("validation accuracy (%)")
for ax in axes:
    ax.set_xticks(range(2, len(h_mlp["acc"]) + 1, 2))
    ax.legend(frameon=False)
plt.tight_layout()
plt.show()

a_mlp, a_cnn = h_mlp["acc"][-1], h_cnn["acc"][-1]
print(f"  MLP {a_mlp:.2f}%   CNN {a_cnn:.2f}%   gap {a_cnn - a_mlp:+.2f} points")
print(f"  error rate {100 - a_mlp:.1f}% -> {100 - a_cnn:.1f}%, "
      f"a {100 * (1 - (100 - a_cnn) / (100 - a_mlp)):.0f}% reduction")

Read the gap as an error rate, not as an accuracy. Going from 87.5% to 92.7%
sounds like five points of polish. Going from a 12.5% error rate to a 7.3% one
means **four of every ten mistakes are gone**, at the same parameter count, on
the same data, in the same number of epochs.

What bought that is locality and weight sharing. The MLP has to learn that the
pixel at `(4, 5)` sits next to the pixel at `(4, 6)`, from data. The
convolution is told.

## 3d · The BatchNorm trap

While the CNN trained, `evaluate` was called twice each epoch on the same 2,000
images with the same weights: once in `eval()` mode and once in `train()` mode.
The only difference between the two curves is which BatchNorm statistics were
used.

In [ ]:
tail = slice(-9, None)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(ep, h_cnn["acc"], "-o", ms=4, color="#a8281f", label="eval() mode, running statistics")
ax.plot(ep, h_cnn["acc_train_mode"], "-o", ms=4, color="#2980b9", label="train() mode, batch statistics")
ax.set_xlabel("epoch"); ax.set_ylabel("validation accuracy (%)")
ax.set_xticks(range(2, len(h_cnn["acc"]) + 1, 2))
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

swing_eval = max(h_cnn["acc"][tail]) - min(h_cnn["acc"][tail])
swing_train = max(h_cnn["acc_train_mode"][tail]) - min(h_cnn["acc_train_mode"][tail])
print(f"  over the last 9 epochs, eval()  moves {swing_eval:.1f} points")
print(f"                          train() moves {swing_train:.1f} points")

Same weights, same images. Only `eval()` wobbles, because only `eval()` uses
the running estimates, and those lag the weights.

If your validation accuracy is wild while training accuracy is smooth, suspect
the BatchNorm statistics before you suspect the model. The cosine schedule in
this run is what settles them: the weights stop moving, so the running averages
catch up.

---

# Part 4 · Augmentation, measured

Data augmentation means applying random label-preserving changes to the
training images so the model sees a slightly different version each epoch.
Thursday's deck gave the standard claim: it acts like more data, so it should
help most when you have little.

You are going to test that claim on 3,000 images, a quarter of what Part 3
used, to give it the best possible chance.

## 4a · Two pipelines, not one

Augmentation goes on the **training** set only. The validation pipeline must be
deterministic, because a score you cannot reproduce is not a measurement, and
because at test time you do not get to see several random versions of an input.

In [ ]:
# TODO: build the two pipelines.
#   train_tf: a v2.Compose of
#       v2.RandomCrop(28, padding=2)   shift the image by up to 2 pixels
#       v2.RandomHorizontalFlip()      mirror it half the time
#   eval_tf: None. The evaluation pipeline does nothing random, and the
#       Dataset already handles the cast and the scaling.

train_tf = None
eval_tf = None

print(train_tf)

In [ ]:
# One image, eight draws from the training pipeline. The first panel is the
# original, so you can see what the augmentation is actually doing.
torch.manual_seed(SEED)
img = x_train[7].unsqueeze(0)

fig, axes = plt.subplots(1, 9, figsize=(13, 2.3))
axes[0].imshow(img[0], cmap="gray_r")
axes[0].set_title("original", fontsize=8)
for ax in axes:
    ax.axis("off"); ax.grid(False)
for ax in axes[1:]:
    ax.imshow(train_tf(img)[0], cmap="gray_r")
plt.suptitle(f"class: {CLASSES[y_train[7]]}", fontsize=9)
plt.tight_layout()
plt.show()

## 4b · Does it help?

Three runs on the same 3,000 images: no augmentation, the crop and flip
pipeline, and a third one that is deliberately too aggressive.

In [ ]:
x_small, y_small = x_train[:3_000], y_train[:3_000]

heavy_tf = v2.Compose([
    v2.RandomResizedCrop(28, scale=(0.2, 1.0), antialias=True),
    v2.RandomRotation(60),
])

runs, models = {}, {}
for name, tf in [("no augmentation", None),
                 ("crop + flip", train_tf),
                 ("aggressive crop + rotate", heavy_tf)]:
    models[name], h = train(build_cnn(), FashionImages(x_small, y_small, tf), val_ds)
    runs[name] = h["acc"][-1]
    print(f"  {name:26s} {h['acc'][-1]:5.2f}%   ({h['seconds']:.0f}s)")

base = runs["no augmentation"]
for name in ("crop + flip", "aggressive crop + rotate"):
    print(f"  {name:26s} {runs[name] - base:+.2f} points against no augmentation")

Both augmented runs are **worse**, and the aggressive one is far worse. That is
not a bug in your code and it is not a bad seed. It is a real property of this
dataset, and it is more useful to you than the result you expected.

## 4c · Why

Two reasons, and you can measure both.

**Fashion-MNIST images are centered and tightly framed.** Every image was
produced the same way, with the garment filling the frame. Shifting it by two
pixels manufactures a kind of image the validation set never contains.

**The footwear classes all point the same way.** A horizontal flip is only
label-preserving if the class really is symmetric. Compare each image to its
own mirror image and the asymmetric classes are obvious.

In [ ]:
asym = []
for k in range(10):
    sel = x_train[y_train == k].float()
    asym.append(((sel - sel.flip(-1)).abs().mean() / sel.mean()).item())

order = np.argsort(asym)[::-1]
fig, ax = plt.subplots(figsize=(8, 3.4))
cols = ["#a8281f" if asym[k] > 0.6 else "#7f8c8d" for k in order]
ax.bar([CLASSES[k] for k in order], [asym[k] for k in order], color=cols)
ax.set_ylabel("mirror mismatch")
ax.tick_params(axis="x", rotation=40)
plt.setp(ax.get_xticklabels(), ha="right")
plt.tight_layout()
plt.show()

print("  most asymmetric classes:")
for k in order[:3]:
    print(f"    {CLASSES[k]:12s} {asym[k]:.3f}")
print("  least asymmetric classes:")
for k in order[-3:]:
    print(f"    {CLASSES[k]:12s} {asym[k]:.3f}")

The three footwear classes sit far above everything else, because every shoe in
Fashion-MNIST faces the same direction. Mirroring one produces a shoe that does
not occur in the data, so the model spends capacity on it for nothing.

## 4d · What augmentation actually buys

So augmentation is useless? No. It buys **robustness to the variation you train
for**. The clean validation set has no shifts in it, so training for shifts
could only cost you. Put shifts into the validation set and the ranking flips.

Nothing is retrained here. These are the two models from 4b, evaluated on the
same images shifted by whole pixels.

In [ ]:
def shift(images, dx, dy):
    """Translate a uint8 batch (N, 28, 28) by whole pixels, zero padded."""
    padded = F.pad(images, (4, 4, 4, 4))
    return torch.roll(padded, shifts=(dy, dx), dims=(-2, -1))[..., 4:-4, 4:-4]


plain_model = models["no augmentation"]
aug_model = models["crop + flip"]

# One knob, five settings: shift down and right by the same amount, so the only
# thing changing is how far the object moves. Note that train_tf only ever
# shifted by 2 pixels, so the last two columns are outside what the augmented
# model trained on as well.
rows = []
for d in range(5):
    vl = DataLoader(FashionImages(shift(x_val, d, d), y_val), batch_size=512)
    a0 = evaluate(plain_model, vl)[0]
    a1 = evaluate(aug_model, vl)[0]
    rows.append((d, a0, a1))
    print(f"  shifted {d}px right and down   plain {a0:5.2f}%   "
          f"augmented {a1:5.2f}%   {a1-a0:+6.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
xs = [r[0] for r in rows]
ax.plot(xs, [r[1] for r in rows], "-o", color="#7f8c8d", lw=2.4, label="no augmentation")
ax.plot(xs, [r[2] for r in rows], "-o", color="#1c7a3e", lw=2.4, label="crop + flip")
ax.set_xticks(xs)
ax.set_xlabel("validation images shifted right and down by (pixels)")
ax.set_ylabel("validation accuracy (%)")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

drop_plain = rows[0][1] - rows[-1][1]
drop_aug = rows[0][2] - rows[-1][2]
print(f"  from 0px to {rows[-1][0]}px:")
print(f"    no augmentation falls {drop_plain:.2f} points")
print(f"    crop + flip     falls {drop_aug:.2f} points")

The augmented model is nearly flat. The plain one falls off a cliff, because it
learned where in the frame to look rather than what to look for. Note that the
last two columns are shifts larger than the 2 pixels `train_tf` ever applied,
so the augmented model is outside its training distribution there too.

That is the whole rule, and it is worth more than any list of transforms:
**augment for the variation your test data actually has.** If your images are
centered, do not train for shifts. If they could arrive off center, rotated or
differently lit, train for exactly that. The torchvision policies from Thursday
(`AutoAugment`, `RandAugment`, `TrivialAugmentWide`) are searched on ImageNet,
where photographs really do vary in all those ways, which is why they transfer
well to natural images and badly to datasets like this one.

---

# Part 5 · What this lab showed

Eight questions. If you can answer them, you are ready for Quiz 6.

1. **Given `Conv2d(C_in, C_out, k, padding=p, stride=s)` and an input size**,
   what comes out? You did this three times in 1a without running anything.
2. **Why a convolution's parameter count does not depend on the image size**,
   and what that buys over `nn.Linear`.
3. **What the receptive field is**, and why a network needs depth to see the
   whole image.
4. **Why the block is conv, then norm, then activation**, and not some other
   order.
5. **What `AdaptiveAvgPool2d(1)` replaced**, how many parameters it saved here,
   and the second thing it bought besides the parameter count.
6. **Why the CNN beat the MLP** at the same parameter count on the same data.
   The answer is two words and neither of them is "capacity".
7. **Why only the `eval()` curve wobbled** in 3d, and what the cosine schedule
   had to do with settling it.
8. **Why augmentation made things worse** in 4b, and what changed in 4d that
   made the same augmentation worth having.

---

# Part 6 · AI disclosure

Required if you used any generative AI on this lab. Name the tool, say where you
used it, and say what for. "None" is a perfectly good answer.

*Tool(s):*

*Where:*

*What for:*

---

# Submitting

1. **Restart the kernel and Run All.** Confirm it runs top to bottom.
2. Check that every plot and every printed block is visible.
3. Render to **HTML or PDF** with resources embedded.
4. Submit the rendered file on Canvas. Due **Wed Oct 7, 11:59 PM ET**.

The notebook is graded for completion, so this is mostly a formality. Part 5 is
your study guide for the quiz that assesses the understanding.

::: {.callout-note}
## Do not submit the data folder
Running this notebook creates a `data/` directory of about 82 MB next to it.
Submit the rendered HTML or PDF, not the folder.
:::